In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q pycocotools opencv-python-headless

import os, cv2, json, shutil, random, glob
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as patches

print("Ready")

Mounted at /content/drive
Ready


In [ ]:
BASE = '/content/drive/MyDrive/FYP/dataset'

# 26 classes outdoor dataset
OUTDOOR_TRAIN = f'{BASE}/26classes/train'
OUTDOOR_VAL   = f'{BASE}/26classes/valid'
OUTDOOR_TEST  = f'{BASE}/26classes/test'

# Indoor DatasetNinja
INDOOR_TRAIN  = f'{BASE}/indoor-objects-detection-DatasetNinja/train'
INDOOR_VAL    = f'{BASE}/indoor-objects-detection-DatasetNinja/valid'
INDOOR_TEST   = f'{BASE}/indoor-objects-detection-DatasetNinja/test'

# MOTS
MOTS_IMAGES   = f'{BASE}/MOTSChallenge/train/images'
MOTS_ANN_TXT  = f'{BASE}/MOTSChallenge/train/instances_txt'

import shutil

OUTPUT = '/content/drive/MyDrive/FYP/fyp-preprocessed'

# Delete and recreate fresh each run
if os.path.exists(OUTPUT):
    print("Deleting existing fyp-preprocessed folder...")
    shutil.rmtree(OUTPUT)
    print("Deleted.")

for split in ['train', 'val', 'test']:
    os.makedirs(f'{OUTPUT}/{split}/images', exist_ok=True)
    os.makedirs(f'{OUTPUT}/{split}/labels', exist_ok=True)

print("Fresh fyp-preprocessed folder created")

print("Paths set. Checking they exist...")
for path in [OUTDOOR_TRAIN, INDOOR_TRAIN, MOTS_IMAGES, MOTS_ANN_TXT]:
    exists = os.path.exists(path)
    print(f"  {'✅' if exists else '❌'} {path}")

Deleting existing fyp-preprocessed folder...
Deleted.
Fresh fyp-preprocessed folder created
Paths set. Checking they exist...
  ✅ /content/drive/MyDrive/FYP/dataset/26classes/train
  ✅ /content/drive/MyDrive/FYP/dataset/indoor-objects-detection-DatasetNinja/train
  ✅ /content/drive/MyDrive/FYP/dataset/MOTSChallenge/train/images
  ✅ /content/drive/MyDrive/FYP/dataset/MOTSChallenge/train/instances_txt


In [ ]:
NAVIGATION_CLASSES = {
    'person':        0,
    'bicycle':       1,
    'car':           2,
    'motorcycle':    3,
    'bus':           4,
    'truck':         5,
    'traffic light': 6,
    'stop sign':     7,
    'bench':         8,
    'chair':         9,
    'couch':         10,
    'table':         11,
    'door':          12,
    'window':        13,
    'cabinet':       14,
    'pole':          15,
    'stairs':        16,
    'dog':           17,
    'cat':           18,
    'fire hydrant':  19,
    'object':        20,   # unknown but relevant obstacle
}

# Classes to completely skip — irrelevant or harmful
SKIP_CLASSES = {
    'gun', 'Bushes', 'branch', 'rat', 'sheep',
    'boat', 'clock', 'suitcase', 'scooter',
    'backpack', 'handbag', 'crosswalk',
    'pothole', 'umbrella', 'elevator',
    'cars-bikes-people', 'tree'
}

print(f"Navigation classes: {len(NAVIGATION_CLASSES)}")
for idx, name in sorted((v,k) for k,v in NAVIGATION_CLASSES.items()):
    print(f"  {idx}: {name}")

Navigation classes: 21
  0: person
  1: bicycle
  2: car
  3: motorcycle
  4: bus
  5: truck
  6: traffic light
  7: stop sign
  8: bench
  9: chair
  10: couch
  11: table
  12: door
  13: window
  14: cabinet
  15: pole
  16: stairs
  17: dog
  18: cat
  19: fire hydrant
  20: object


In [ ]:
def check_black_boxes(img, threshold=0.08):
    """
    Detect images with large black rectangular patches
    (privacy redactions like in the bus image)
    Returns True if too many black boxes found
    """
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    # Black pixels
    black_mask = (gray < 15).astype(np.uint8) * 255
    black_ratio = np.sum(black_mask > 0) / (img.shape[0] * img.shape[1])

    if black_ratio < 0.02:
        return False  # Very few black pixels, fine

    # Check if black pixels form rectangular blocks (redactions)
    # Use morphological operations to find solid black rectangles
    kernel = np.ones((20, 20), np.uint8)
    dilated = cv2.dilate(black_mask, kernel)
    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    large_black_rects = 0
    img_area = img.shape[0] * img.shape[1]

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area > img_area * 0.02:  # Block covers >2% of image
            x, y, w, h = cv2.boundingRect(cnt)
            rect_area = w * h
            # Check if it's roughly rectangular (solid block)
            if area / rect_area > 0.7:
                large_black_rects += 1

    return large_black_rects >= 2  # 2+ large black rectangles = redacted


def estimate_viewpoint(img):
    """
    Estimate if image is from a valid wearable camera viewpoint.

    Detects and rejects:
    1. Top-down / bird's eye view (drone/aerial shots like the car top-down image)
    2. Extreme close-up (object fills >85% of frame, like drawer handle image)
    3. Extreme upward angle (like the traffic lights shot upward)
    4. Studio/product shots (white/plain background, single centred object)

    Returns (is_valid, reason)
    """
    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # ── 1. White/plain background studio shot ──────────────────────────────
    white_ratio = np.sum(gray > 230) / (h * w)
    if white_ratio > 0.50:
        return False, "studio/white background"

    # ── 2. Check for extreme close-up ─────────────────────────────────────
    # In extreme close-ups, there is very little edge variation
    # because the object fills the frame with a uniform texture
    edges = cv2.Canny(gray, 50, 150)
    edge_density = np.sum(edges > 0) / (h * w)

    # Very low edge density in a non-plain image = likely extreme close-up
    # of a textured surface (like wood grain filling the whole frame)
    if edge_density < 0.02 and white_ratio < 0.3:
        return False, "extreme close-up / low detail"

    # ── 3. Detect top-down/bird's eye view ────────────────────────────────
    # In top-down shots, horizontal lines dominate and there is
    # no perspective vanishing point — we check line orientation
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, 50,
                             minLineLength=w*0.15, maxLineGap=20)

    if lines is not None and len(lines) > 5:
        angles = []
        for line in lines:
            x1, y1, x2, y2 = line[0]
            if x2 - x1 == 0:
                angles.append(90)
            else:
                angle = abs(np.degrees(np.arctan2(y2-y1, x2-x1)))
                angles.append(angle)

        angles = np.array(angles)
        # Top-down images have mostly horizontal lines (0-20°)
        # and very few vertical/diagonal lines
        horizontal = np.sum(angles < 20) / len(angles)
        if horizontal > 0.85:
            return False, "top-down/aerial viewpoint"

    # ── 4. Detect extreme upward angle ────────────────────────────────────
    # In upward-angle shots (like traffic lights shot from below),
    # the bottom half of the image is much brighter (sky/ceiling)
    # and the object appears in the upper portion
    top_half_bright = np.mean(gray[:h//2, :])
    bottom_half_bright = np.mean(gray[h//2:, :])

    # If top is much darker than bottom, likely shot upward
    # (sky at top is usually brighter, subject in lower frame)
    # Reversed = upward angle
    if bottom_half_bright > top_half_bright * 1.8 and top_half_bright < 80:
        return False, "extreme upward angle"

    # ── 5. Object too close — fills entire frame ───────────────────────────
    # When object fills frame, corners tend to have same colour as centre
    # and there's very little background visible
    corners = [
        gray[10:30, 10:30],
        gray[10:30, w-30:w-10],
        gray[h-30:h-10, 10:30],
        gray[h-30:h-10, w-30:w-10]
    ]
    centre = gray[h//2-20:h//2+20, w//2-20:w//2+20]

    corner_means = [np.mean(c) for c in corners]
    centre_mean = np.mean(centre)

    # If all corners are similar brightness to centre, no background visible
    corner_diffs = [abs(cm - centre_mean) for cm in corner_means]
    if all(d < 25 for d in corner_diffs) and edge_density < 0.06:
        return False, "object too close / fills frame"

    return True, "ok"


def is_valid_image(img_path):
    """
    Full image validation combining all checks
    Returns (is_valid, reason)
    """
    try:
        img = cv2.imread(str(img_path))
        if img is None:
            return False, "corrupted"

        h, w = img.shape[:2]
        if h < 96 or w < 96:
            return False, f"too small {w}x{h}"

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        brightness = np.mean(gray)

        if brightness < 15:
            return False, f"too dark {brightness:.1f}"

        blur = cv2.Laplacian(gray, cv2.CV_64F).var()
        if blur < 5:
            return False, f"too blurry {blur:.1f}"

        # Check black box redactions
        if check_black_boxes(img):
            return False, "privacy redaction boxes"

        # Check viewpoint
        valid, reason = estimate_viewpoint(img)
        if not valid:
            return False, reason

        return True, "ok"

    except Exception as e:
        return False, str(e)


def resize_pad(img, size=320):
    h, w = img.shape[:2]
    scale = size / max(h, w)
    nh, nw = int(h * scale), int(w * scale)
    resized = cv2.resize(img, (nw, nh))
    ph, pw = size - nh, size - nw
    return cv2.copyMakeBorder(
        resized, ph//2, ph-ph//2, pw//2, pw-pw//2,
        cv2.BORDER_CONSTANT, value=(114,114,114)
    )

print("Validation functions ready")

Validation functions ready


In [ ]:
def process_indoor(split_dir, output_split):
    indoor_to_nav = {
        'door':        'door',
        'openedDoor':  'door',
        'window':      'window',
        'chair':       'chair',
        'table':       'table',
        'cabinet':     'cabinet',
        'couch':       'couch',
        'pole':        'pole',
    }

    img_dir = Path(split_dir) / 'img'
    ann_dir = Path(split_dir) / 'ann'

    if not img_dir.exists():
        print(f"  ❌ Not found: {img_dir}")
        return 0

    img_files = list(img_dir.glob('*.png')) + list(img_dir.glob('*.jpg'))
    saved = 0
    skip_reasons = {}

    for img_path in img_files:
        ann_path = ann_dir / (img_path.name + '.json')
        if not ann_path.exists():
            skip_reasons['no annotation'] = skip_reasons.get('no annotation', 0) + 1
            continue

        valid, reason = is_valid_image(img_path)
        if not valid:
            skip_reasons[reason] = skip_reasons.get(reason, 0) + 1
            continue

        with open(ann_path) as f:
            ann = json.load(f)

        iw = ann['size']['width']
        ih = ann['size']['height']
        labels = []

        for obj in ann['objects']:
            title = obj['classTitle']
            if title not in indoor_to_nav:
                continue
            nav = indoor_to_nav[title]
            cls_id = NAVIGATION_CLASSES[nav]

            pts = obj['points']['exterior']
            x1, y1 = pts[0]
            x2, y2 = pts[1]

            cx = ((x1+x2)/2) / iw
            cy = ((y1+y2)/2) / ih
            bw = abs(x2-x1) / iw
            bh = abs(y2-y1) / ih

            if bw < 0.01 or bh < 0.01:
                continue

            # Skip objects that are too small (far away)
            # or too large (too close, fills frame)
            obj_area = bw * bh
            if obj_area < 0.002:  # Object <0.2% of image = too far
                continue
            if obj_area > 0.90:   # Object >90% of image = too close
                continue

            cx = min(max(cx,0),1)
            cy = min(max(cy,0),1)
            bw = min(bw,1)
            bh = min(bh,1)
            labels.append(f"{cls_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")

        if not labels:
            skip_reasons['no valid labels'] = skip_reasons.get('no valid labels', 0) + 1
            continue

        img = cv2.imread(str(img_path))
        img_r = resize_pad(img)
        fname = f"indoor_{img_path.stem}.jpg"
        cv2.imwrite(f"{OUTPUT}/{output_split}/images/{fname}", img_r)
        with open(f"{OUTPUT}/{output_split}/labels/{fname.replace('.jpg','.txt')}", 'w') as f:
            f.write('\n'.join(labels))
        saved += 1

    total_skipped = sum(skip_reasons.values())
    print(f"  {output_split}: saved={saved}, skipped={total_skipped}")
    print(f"  Skip breakdown: {skip_reasons}")
    return saved


print("Processing Indoor dataset...")
t = process_indoor(INDOOR_TRAIN, 'train')
v = process_indoor(INDOOR_VAL,   'val')
e = process_indoor(INDOOR_TEST,  'test')
print(f"Indoor total: {t+v+e}")

Processing Indoor dataset...
  train: saved=471, skipped=508
  Skip breakdown: {'top-down/aerial viewpoint': 43, 'no valid labels': 296, 'extreme close-up / low detail': 95, 'extreme upward angle': 4, 'privacy redaction boxes': 32, 'studio/white background': 27, 'object too close / fills frame': 8, 'too blurry 4.8': 1, 'too blurry 3.3': 1, 'too blurry 4.7': 1}
  val: saved=99, skipped=123
  Skip breakdown: {'too dark 12.2': 1, 'privacy redaction boxes': 9, 'no valid labels': 81, 'extreme close-up / low detail': 12, 'studio/white background': 8, 'top-down/aerial viewpoint': 7, 'too blurry 2.4': 1, 'too blurry 3.7': 1, 'object too close / fills frame': 3}
  test: saved=83, skipped=24
  Skip breakdown: {'privacy redaction boxes': 12, 'object too close / fills frame': 1, 'top-down/aerial viewpoint': 4, 'extreme close-up / low detail': 3, 'no valid labels': 4}
Indoor total: 653


In [ ]:
def process_outdoor_coco(split_dir, output_split, max_images=10000):
    split_path = Path(split_dir)
    ann_file = split_path / '_annotations.coco.json'

    if not ann_file.exists():
        print(f"  ❌ Not found: {ann_file}")
        return 0

    outdoor_to_nav = {
        'person':        'person',
        'Person':        'person',
        'bicycle':       'bicycle',
        'car':           'car',
        'motorcycle':    'motorcycle',
        'Bus':           'bus',
        'bus':           'bus',
        'Truck':         'truck',
        'truck':         'truck',
        'bench':         'bench',
        'chair':         'chair',
        'door':          'door',
        'stairs':        'stairs',
        'stop_sign':     'stop sign',
        'fire_hydrant':  'fire hydrant',
        'traffic light': 'traffic light',
        'green_light':   'traffic light',
        'red_light':     'traffic light',
        'yellow_light':  'traffic light',
        'traffic_cone':  'object',
        'train':         'object',
    }

    print(f"  Loading annotations...")
    with open(ann_file) as f:
        coco = json.load(f)

    id_to_file = {img['id']: img['file_name'] for img in coco['images']}
    id_to_cat  = {cat['id']: cat['name'] for cat in coco['categories']}

    img_anns = {}
    for ann in coco['annotations']:
        img_id = ann['image_id']
        if img_id not in img_anns:
            img_anns[img_id] = []
        img_anns[img_id].append(ann)

    all_ids = list(img_anns.keys())
    random.shuffle(all_ids)
    all_ids = all_ids[:max_images]

    print(f"  Processing {len(all_ids)} images...")

    saved = 0
    skip_reasons = {}

    for i, img_id in enumerate(all_ids):
        fname = id_to_file.get(img_id)
        if not fname:
            skip_reasons['no filename'] = skip_reasons.get('no filename', 0) + 1
            continue

        img_path = split_path / fname
        if not img_path.exists():
            skip_reasons['file missing'] = skip_reasons.get('file missing', 0) + 1
            continue

        # Full validation including viewpoint check
        valid, reason = is_valid_image(img_path)
        if not valid:
            skip_reasons[reason] = skip_reasons.get(reason, 0) + 1
            continue

        img = cv2.imread(str(img_path))
        if img is None:
            skip_reasons['cv2 read fail'] = skip_reasons.get('cv2 read fail', 0) + 1
            continue

        ih, iw = img.shape[:2]
        labels = []

        for ann in img_anns[img_id]:
            cat_name = id_to_cat.get(ann['category_id'], '')

            if cat_name in SKIP_CLASSES:
                continue

            nav_name = outdoor_to_nav.get(cat_name, 'object')
            cls_id = NAVIGATION_CLASSES[nav_name]

            x, y, bw, bh = ann['bbox']
            cx = (x + bw/2) / iw
            cy = (y + bh/2) / ih
            nw = bw / iw
            nh = bh / ih

            # Filter objects too small (too far) or too large (too close)
            obj_area = nw * nh
            if obj_area < 0.002:
                continue
            if obj_area > 0.90:
                continue

            if nw < 0.01 or nh < 0.01:
                continue

            cx = min(max(cx,0),1)
            cy = min(max(cy,0),1)
            nw = min(nw,1)
            nh = min(nh,1)
            labels.append(f"{cls_id} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")

        if not labels:
            skip_reasons['no valid labels'] = skip_reasons.get('no valid labels', 0) + 1
            continue

        img_r = resize_pad(img)
        out_fname = f"outdoor_{Path(fname).stem}.jpg"
        cv2.imwrite(f"{OUTPUT}/{output_split}/images/{out_fname}", img_r)
        with open(f"{OUTPUT}/{output_split}/labels/{out_fname.replace('.jpg','.txt')}", 'w') as f:
            f.write('\n'.join(labels))
        saved += 1

        if (i+1) % 500 == 0:
            print(f"    {i+1}/{len(all_ids)} — saved: {saved}")

    print(f"  {output_split}: saved={saved}, skipped={sum(skip_reasons.values())}")
    print(f"  Skip breakdown: {skip_reasons}")
    return saved


print("Processing Outdoor dataset...")
t = process_outdoor_coco(OUTDOOR_TRAIN, 'train', max_images=10000)
v = process_outdoor_coco(OUTDOOR_VAL,   'val',   max_images=1500)
e = process_outdoor_coco(OUTDOOR_TEST,  'test',  max_images=800)
print(f"Outdoor total: {t+v+e}")

Processing Outdoor dataset...
  Loading annotations...
  Processing 10000 images...
    500/10000 — saved: 286
    2000/10000 — saved: 1130
    2500/10000 — saved: 1402
    3000/10000 — saved: 1670
    3500/10000 — saved: 1960
    4000/10000 — saved: 2248
    4500/10000 — saved: 2531
    5000/10000 — saved: 2813
    6500/10000 — saved: 3689
    8000/10000 — saved: 4550
    9500/10000 — saved: 5383
  train: saved=5675, skipped=4325
  Skip breakdown: {'no valid labels': 2974, 'studio/white background': 647, 'extreme upward angle': 273, 'privacy redaction boxes': 229, 'top-down/aerial viewpoint': 146, 'object too close / fills frame': 19, 'too dark 5.8': 1, 'extreme close-up / low detail': 28, 'too dark 12.0': 1, 'too dark 12.9': 1, 'too dark 14.9': 1, 'too dark 13.4': 1, 'too dark 8.2': 1, 'too dark 7.3': 1, 'too dark 14.7': 1, 'too dark 14.0': 1}
  Loading annotations...
  Processing 1500 images...
    500/1500 — saved: 269
    1500/1500 — saved: 793
  val: saved=793, skipped=707
  Skip

In [ ]:
def decode_rle_bbox(rle_str, height, width):
    """Decode MOTS RLE to YOLO bounding box"""
    try:
        from pycocotools import mask as mask_util
        rle = {'counts': rle_str.encode(), 'size': [height, width]}
        decoded = mask_util.decode(rle)
        rows = np.any(decoded, axis=1)
        cols = np.any(decoded, axis=0)
        if not rows.any() or not cols.any():
            return None
        y1, y2 = np.where(rows)[0][[0,-1]]
        x1, x2 = np.where(cols)[0][[0,-1]]
        cx = ((x1+x2)/2) / width
        cy = ((y1+y2)/2) / height
        bw = (x2-x1) / width
        bh = (y2-y1) / height
        return cx, cy, bw, bh
    except:
        return None


def process_mots(images_dir, ann_txt_dir, max_per_seq=200):
    """
    MOTS: images in images/0002,0005,0009,0011
    Annotations: instances_txt/0002.txt etc
    Class 2=person, Class 10=car
    """
    sequences = ['0002', '0005', '0009', '0011']
    total = 0

    for seq in sequences:
        img_seq_dir = Path(images_dir) / seq
        ann_file = Path(ann_txt_dir) / f'{seq}.txt'

        if not img_seq_dir.exists() or not ann_file.exists():
            print(f"  Sequence {seq} not found")
            continue

        print(f"  Processing sequence {seq}...")

        # Parse annotations
        frame_anns = {}
        with open(ann_file) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 6:
                    continue
                frame_id = int(parts[0])
                cls_id = int(parts[2])
                h = int(parts[3])
                w = int(parts[4])
                rle = parts[5]

                if cls_id == 2:
                    nav_cls = 0   # person
                elif cls_id == 10:
                    nav_cls = 2   # car
                else:
                    continue

                bbox = decode_rle_bbox(rle, h, w)
                if bbox is None:
                    continue

                if frame_id not in frame_anns:
                    frame_anns[frame_id] = []
                frame_anns[frame_id].append((nav_cls, *bbox))

        img_files = sorted(
            list(img_seq_dir.glob('*.png')) +
            list(img_seq_dir.glob('*.jpg'))
        )[:max_per_seq]

        saved = skipped = 0
        for img_path in img_files:
            frame_id = int(img_path.stem)
            if frame_id not in frame_anns:
                skipped += 1
                continue

            valid, reason = is_valid_image(img_path)
            if not valid:
                skipped += 1
                continue

            img = cv2.imread(str(img_path))
            if img is None:
                skipped += 1
                continue

            labels = []
            for ann in frame_anns[frame_id]:
                cls, cx, cy, bw, bh = ann
                if bw > 0.01 and bh > 0.01:
                    labels.append(f"{cls} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")

            if not labels:
                skipped += 1
                continue

            img_r = resize_pad(img)
            fname = f"mots_{seq}_{img_path.stem}.jpg"
            cv2.imwrite(f"{OUTPUT}/train/images/{fname}", img_r)
            with open(f"{OUTPUT}/train/labels/{fname.replace('.jpg','.txt')}", 'w') as f:
                f.write('\n'.join(labels))

            saved += 1

        print(f"    seq {seq}: saved={saved}, skipped={skipped}")
        total += saved

    print(f"MOTS total: {total}")
    return total


print("Processing MOTS dataset...")
mots_total = process_mots(MOTS_IMAGES, MOTS_ANN_TXT)

Processing MOTS dataset...
  Processing sequence 0002...
    seq 0002: saved=55, skipped=145
  Processing sequence 0005...
    seq 0005: saved=199, skipped=1
  Processing sequence 0009...
    seq 0009: saved=111, skipped=89
  Processing sequence 0011...
    seq 0011: saved=195, skipped=5
MOTS total: 560


In [ ]:
train_n = len(os.listdir(f'{OUTPUT}/train/images'))
val_n   = len(os.listdir(f'{OUTPUT}/val/images'))
test_n  = len(os.listdir(f'{OUTPUT}/test/images'))

print("=" * 40)
print("PREPROCESSING COMPLETE")
print("=" * 40)
print(f"Train : {train_n}")
print(f"Val   : {val_n}")
print(f"Test  : {test_n}")
print(f"Total : {train_n + val_n + test_n}")
print("=" * 40)

yaml = f"""path: {OUTPUT}
train: train/images
val: val/images
test: test/images
nc: {len(NAVIGATION_CLASSES)}
names:
"""
for idx, name in sorted((v,k) for k,v in NAVIGATION_CLASSES.items()):
    yaml += f"  {idx}: {name}\n"

with open(f'{OUTPUT}/dataset.yaml', 'w') as f:
    f.write(yaml)

print("\ndataset.yaml saved")
print(yaml)

PREPROCESSING COMPLETE
Train : 6706
Val   : 892
Test  : 479
Total : 8077

dataset.yaml saved
path: /content/drive/MyDrive/FYP/fyp-preprocessed
train: train/images
val: val/images
test: test/images
nc: 21
names:
  0: person
  1: bicycle
  2: car
  3: motorcycle
  4: bus
  5: truck
  6: traffic light
  7: stop sign
  8: bench
  9: chair
  10: couch
  11: table
  12: door
  13: window
  14: cabinet
  15: pole
  16: stairs
  17: dog
  18: cat
  19: fire hydrant
  20: object



In [ ]:
def show_extended_validation(n_per_category=3):
    """
    Shows before/after with clear labels on why images were kept or removed.
    Categories: indoor, outdoor, people, multi-object, different distances
    """
    class_names = {v: k for k, v in NAVIGATION_CLASSES.items()}

    def draw_boxes(img, label_path):
        img_draw = img.copy()
        h, w = img_draw.shape[:2]
        if os.path.exists(label_path):
            with open(label_path) as f:
                for line in f:
                    p = line.strip().split()
                    if len(p) == 5:
                        cls = int(p[0])
                        cx,cy,bw,bh = float(p[1]),float(p[2]),float(p[3]),float(p[4])
                        x1 = int((cx-bw/2)*w)
                        y1 = int((cy-bh/2)*h)
                        x2 = int((cx+bw/2)*w)
                        y2 = int((cy+bh/2)*h)
                        cv2.rectangle(img_draw, (x1,y1), (x2,y2), (0,255,0), 2)
                        label = class_names.get(cls, str(cls))
                        cv2.putText(img_draw, label, (x1, max(y1-5,10)),
                                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)
        return img_draw

    # ── SECTION 1: Retained images ─────────────────────────────────────────
    print("=" * 60)
    print("SECTION 1: RETAINED IMAGES (kept after preprocessing)")
    print("=" * 60)

    for split, label in [('train','TRAIN'), ('val','VAL')]:
        img_dir = f'{OUTPUT}/{split}/images'
        lbl_dir = f'{OUTPUT}/{split}/labels'
        files = os.listdir(img_dir)
        if not files:
            continue

        # Categorise by prefix
        indoor_files  = [f for f in files if f.startswith('indoor')]
        outdoor_files = [f for f in files if f.startswith('outdoor')]
        mots_files    = [f for f in files if f.startswith('mots')]

        for category, cat_files, cat_name in [
            (indoor_files,  indoor_files,  'Indoor'),
            (outdoor_files, outdoor_files, 'Outdoor'),
            (mots_files,    mots_files,    'MOTS (People)'),
        ]:
            if not cat_files:
                continue

            samples = random.sample(cat_files, min(n_per_category, len(cat_files)))
            fig, axes = plt.subplots(1, len(samples), figsize=(6*len(samples), 5))
            if len(samples) == 1:
                axes = [axes]

            fig.suptitle(f'✅ RETAINED — {cat_name} ({label})',
                        fontsize=13, color='green', weight='bold')

            for i, fname in enumerate(samples):
                img = cv2.imread(f'{img_dir}/{fname}')
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                lbl = f'{lbl_dir}/{fname.replace(".jpg",".txt")}'
                img_boxes = draw_boxes(img, lbl)

                # Count objects
                obj_count = 0
                obj_types = []
                if os.path.exists(lbl):
                    with open(lbl) as f:
                        lines = f.readlines()
                        obj_count = len(lines)
                        obj_types = list(set([
                            class_names.get(int(l.split()[0]),'?')
                            for l in lines if l.strip()
                        ]))

                axes[i].imshow(img_boxes)
                axes[i].set_title(
                    f'{fname[:25]}\nObjects: {obj_count} | {", ".join(obj_types[:3])}',
                    fontsize=7
                )
                axes[i].axis('off')

            plt.tight_layout()
            plt.show()

    # ── SECTION 2: Removed images with reasons ─────────────────────────────
    print("\n" + "=" * 60)
    print("SECTION 2: REMOVED IMAGES (with reason for removal)")
    print("=" * 60)

    # Test some of the actual problematic image types we know about
    removal_examples = [
        # (image_path, expected_reason)
    ]

    # Collect some images from datasets to test and show why removed
    test_sources = []
    for split_dir in [OUTDOOR_TRAIN]:
        files = [f for f in os.listdir(split_dir)
                 if f.endswith(('.jpg','.png'))]
        test_sources.extend([
            os.path.join(split_dir, f)
            for f in random.sample(files, min(50, len(files)))
        ])

    removed = []
    for img_path in test_sources:
        valid, reason = is_valid_image(img_path)
        if not valid:
            removed.append((img_path, reason))
        if len(removed) >= 9:
            break

    if removed:
        n = min(9, len(removed))
        cols = 3
        rows = (n + cols - 1) // cols
        fig, axes = plt.subplots(rows, cols, figsize=(6*cols, 5*rows))
        axes = axes.flatten() if rows > 1 else axes
        if n == 1:
            axes = [axes]

        fig.suptitle('❌ REMOVED IMAGES — Reason shown in title',
                    fontsize=13, color='red', weight='bold')

        for i, (img_path, reason) in enumerate(removed[:n]):
            img = cv2.imread(img_path)
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                axes[i].imshow(img)
            axes[i].set_title(
                f'REMOVED: {reason}\n{Path(img_path).name[:30]}',
                fontsize=8, color='red'
            )
            axes[i].axis('off')

        # Hide unused subplots
        for j in range(n, len(axes)):
            axes[j].axis('off')

        plt.tight_layout()
        plt.show()
    else:
        print("No removed images found in sample — all passed validation")

    # ── SECTION 3: People check ─────────────────────────────────────────────
    print("\n" + "=" * 60)
    print("SECTION 3: PEOPLE DETECTION CHECK")
    print("=" * 60)

    img_dir = f'{OUTPUT}/train/images'
    lbl_dir = f'{OUTPUT}/train/labels'

    person_files = []
    for fname in os.listdir(lbl_dir):
        lbl_path = f'{lbl_dir}/{fname}'
        with open(lbl_path) as f:
            lines = f.readlines()
        if any(int(l.split()[0]) == 0 for l in lines if l.strip()):
            img_fname = fname.replace('.txt', '.jpg')
            if os.path.exists(f'{img_dir}/{img_fname}'):
                person_files.append(img_fname)

    print(f"Images containing people: {len(person_files)}")

    if person_files:
        samples = random.sample(person_files, min(4, len(person_files)))
        fig, axes = plt.subplots(1, len(samples), figsize=(6*len(samples), 5))
        if len(samples) == 1:
            axes = [axes]
        fig.suptitle('👤 PEOPLE DETECTED — Verification',
                    fontsize=13, color='blue', weight='bold')

        for i, fname in enumerate(samples):
            img = cv2.imread(f'{img_dir}/{fname}')
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            lbl = f'{lbl_dir}/{fname.replace(".jpg",".txt")}'
            img_boxes = draw_boxes(img, lbl)
            axes[i].imshow(img_boxes)
            axes[i].set_title(fname[:30], fontsize=7)
            axes[i].axis('off')

        plt.tight_layout()
        plt.show()

    # ── SECTION 4: Multi-object images ──────────────────────────────────────
    print("\n" + "=" * 60)
    print("SECTION 4: MULTI-OBJECT IMAGES")
    print("=" * 60)

    multi_files = []
    for fname in os.listdir(lbl_dir):
        lbl_path = f'{lbl_dir}/{fname}'
        with open(lbl_path) as f:
            lines = [l for l in f.readlines() if l.strip()]
        if len(lines) >= 3:
            img_fname = fname.replace('.txt', '.jpg')
            if os.path.exists(f'{img_dir}/{img_fname}'):
                multi_files.append((img_fname, len(lines)))

    multi_files.sort(key=lambda x: x[1], reverse=True)
    print(f"Images with 3+ objects: {len(multi_files)}")

    if multi_files:
        samples = [f[0] for f in multi_files[:4]]
        fig, axes = plt.subplots(1, len(samples), figsize=(6*len(samples), 5))
        if len(samples) == 1:
            axes = [axes]
        fig.suptitle('📦 MULTI-OBJECT IMAGES',
                    fontsize=13, color='purple', weight='bold')

        for i, fname in enumerate(samples):
            img = cv2.imread(f'{img_dir}/{fname}')
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            lbl = f'{lbl_dir}/{fname.replace(".jpg",".txt")}'
            img_boxes = draw_boxes(img, lbl)

            with open(lbl) as f:
                count = len([l for l in f.readlines() if l.strip()])

            axes[i].imshow(img_boxes)
            axes[i].set_title(f'{fname[:25]}\n{count} objects', fontsize=7)
            axes[i].axis('off')

        plt.tight_layout()
        plt.show()

    # ── SECTION 5: Dataset statistics ───────────────────────────────────────
    print("\n" + "=" * 60)
    print("SECTION 5: DATASET STATISTICS")
    print("=" * 60)

    class_counts = {name: 0 for name in NAVIGATION_CLASSES}
    lbl_dir = f'{OUTPUT}/train/labels'

    for fname in os.listdir(lbl_dir):
        with open(f'{lbl_dir}/{fname}') as f:
            for line in f:
                if line.strip():
                    cls_id = int(line.split()[0])
                    cls_name = class_names.get(cls_id, 'unknown')
                    if cls_name in class_counts:
                        class_counts[cls_name] += 1

    class_counts = {k: v for k, v in class_counts.items() if v > 0}
    sorted_counts = sorted(class_counts.items(), key=lambda x: x[1], reverse=True)

    names = [x[0] for x in sorted_counts]
    counts = [x[1] for x in sorted_counts]

    plt.figure(figsize=(14, 5))
    bars = plt.bar(names, counts, color='steelblue')
    plt.xticks(rotation=45, ha='right')
    plt.title('Object Count per Class in Training Set')
    plt.ylabel('Count')
    for bar, count in zip(bars, counts):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(count), ha='center', va='bottom', fontsize=8)
    plt.tight_layout()
    plt.show()

    print("\nClass distribution:")
    for name, count in sorted_counts:
        print(f"  {name}: {count}")


show_extended_validation(n_per_category=3)

In [ ]:
import os

OUTPUT = '/content/drive/MyDrive/FYP/fyp-preprocessed'

train_imgs  = len(os.listdir(f'{OUTPUT}/train/images'))
train_lbls  = len(os.listdir(f'{OUTPUT}/train/labels'))
val_imgs    = len(os.listdir(f'{OUTPUT}/val/images'))
val_lbls    = len(os.listdir(f'{OUTPUT}/val/labels'))
test_imgs   = len(os.listdir(f'{OUTPUT}/test/images'))
test_lbls   = len(os.listdir(f'{OUTPUT}/test/labels'))

print("=" * 40)
print("PREPROCESSED DATASET SUMMARY")
print("=" * 40)
print(f"Train  — images: {train_imgs}, labels: {train_lbls}")
print(f"Val    — images: {val_imgs},  labels: {val_lbls}")
print(f"Test   — images: {test_imgs}, labels: {test_lbls}")
print(f"Total  — {train_imgs + val_imgs + test_imgs} images")
print("=" * 40)

# Check images match labels
print("\nConsistency check:")
print(f"  Train match : {'✅' if train_imgs == train_lbls else '❌'}")
print(f"  Val match   : {'✅' if val_imgs == val_lbls else '❌'}")
print(f"  Test match  : {'✅' if test_imgs == test_lbls else '❌'}")

PREPROCESSED DATASET SUMMARY
Train  — images: 6706, labels: 6706
Val    — images: 892,  labels: 892
Test   — images: 479, labels: 479
Total  — 8077 images

Consistency check:
  Train match : ✅
  Val match   : ✅
  Test match  : ✅


In [ ]:
import os

OUTPUT = '/content/drive/MyDrive/FYP/fyp-preprocessed'

train_imgs = len(os.listdir(f'{OUTPUT}/train/images'))
train_lbls = len(os.listdir(f'{OUTPUT}/train/labels'))
val_imgs   = len(os.listdir(f'{OUTPUT}/val/images'))
val_lbls   = len(os.listdir(f'{OUTPUT}/val/labels'))
test_imgs  = len(os.listdir(f'{OUTPUT}/test/images'))
test_lbls  = len(os.listdir(f'{OUTPUT}/test/labels'))

print("=" * 40)
print("PREPROCESSED DATASET SUMMARY")
print("=" * 40)
print(f"Train  — images: {train_imgs}, labels: {train_lbls}")
print(f"Val    — images: {val_imgs},  labels: {val_lbls}")
print(f"Test   — images: {test_imgs}, labels: {test_lbls}")
print(f"Total  — {train_imgs + val_imgs + test_imgs} images")
print("=" * 40)

print("\nConsistency check:")
print(f"  Train match : {'✅' if train_imgs == train_lbls else '❌'}")
print(f"  Val match   : {'✅' if val_imgs == val_lbls else '❌'}")
print(f"  Test match  : {'✅' if test_imgs == test_lbls else '❌'}")

# Check dataset.yaml exists
yaml_exists = os.path.exists(f'{OUTPUT}/dataset.yaml')
print(f"  dataset.yaml: {'✅' if yaml_exists else '❌'}")

# Check prefix distribution
prefixes = {'indoor': 0, 'outdoor': 0, 'mots': 0, 'aug': 0, 'other': 0}
for f in os.listdir(f'{OUTPUT}/train/images'):
    matched = False
    for prefix in ['indoor', 'outdoor', 'mots', 'aug']:
        if f.startswith(prefix):
            prefixes[prefix] += 1
            matched = True
            break
    if not matched:
        prefixes['other'] += 1

print(f"\nTrain source breakdown:")
for src, count in prefixes.items():
    if count > 0:
        print(f"  {src}: {count}")

PREPROCESSED DATASET SUMMARY
Train  — images: 6706, labels: 6706
Val    — images: 892,  labels: 892
Test   — images: 479, labels: 479
Total  — 8077 images

Consistency check:
  Train match : ✅
  Val match   : ✅
  Test match  : ✅
  dataset.yaml: ✅

Train source breakdown:
  indoor: 471
  outdoor: 5675
  mots: 560
